# 01 - Build an agent: code, prompt-based and hosted

**Case study:** *Maple Mortgage Assistant* for Northwind Home Lending (fictional lender).

## What you'll learn
- **Agent Framework (AF):** build a client, an `Agent` (name, instructions, tools, context provider), run it, stream it, keep a session
- **Azure AI Projects (`AIProjectClient`):** register a **prompt agent** with Foundry, list/get its versions, call it, delete it
- What a **hosted agent** is, with the 10-line server that makes one
- Which assets Foundry can store, and which stay in your code

Everything is written in the notebook cells, no hidden helpers. Auth is Entra ID only (`az login`, `DefaultAzureCredential`). No keys.

## 1. Setup
The helper module `shared/fsi_common.py` now holds only boilerplate: it reads `day-2/.env` (prints *set/unset*, never values), the Maple system prompt, mock data and a PII `redact`.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `shared` importable (notebook cwd = notebooks/)

# Boilerplate only: env loading, the Maple system prompt, mock mortgage data, PII redaction.
from shared.fsi_common import load_settings, print_env_check, MAPLE_INSTRUCTIONS, redact

settings = load_settings(require=("project_endpoint", "model"))
print_env_check(settings)

AZURE_SUBSCRIPTION_ID                      set
AZURE_RESOURCE_GROUP                       set
AZURE_LOCATION                             set
AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set
APPLICATIONINSIGHTS_CONNECTION_STRING      set
AZURE_CONTAINER_REGISTRY_NAME              set


## 2. Part A - Microsoft Agent Framework (runs in *your* process)
### 2.1 Chat client
`FoundryChatClient` is AF's connection to a model deployment in your Foundry project. `DefaultAzureCredential` picks up your `az login`.

In [2]:
from azure.identity import DefaultAzureCredential
from agent_framework.foundry import FoundryChatClient

credential = DefaultAzureCredential()
client = FoundryChatClient(project_endpoint=settings.project_endpoint, model=settings.model, credential=credential)
print(type(client).__name__, "-> model deployment:", settings.model)

FoundryChatClient -> model deployment: gpt-4.1-mini


### 2.2 The agent
`Agent(client, instructions, name=..., tools=..., context_providers=...)`. Instructions are the system prompt (our compliance rules: no rate promises, always a disclosure, mask PII, escalate when unsure). `run()` is one turn.

In [3]:
from agent_framework import Agent

agent = Agent(client, MAPLE_INSTRUCTIONS, name="maple")
resp = await agent.run("A first-time buyer asks what pre-qualification means. Explain in 3 sentences.")
print(redact(resp.text))
print("[usage]", dict(resp.usage_details or {}))

Pre-qualification is an early step where the lender reviews your financial information to estimate how much you might afford to borrow. It gives you an idea of your budget before you start house hunting. However, it’s not a guarantee of approval or a final loan offer.
[usage] {'openai.cache_write_tokens': 0, 'total_token_count': 239, 'openai.reasoning_tokens': 0, 'input_token_count': 185, 'output_token_count': 54, 'openai.cached_input_tokens': 0, 'cache_read_input_token_count': 0, 'cache_creation_input_token_count': 0, 'reasoning_output_token_count': 0}


### 2.3 Streaming
`run(..., stream=True)` returns an async stream of updates; `get_final_response()` gives the assembled response (with usage). Better UX for a loan officer waiting on an answer.

In [4]:
stream = agent.run("List 3 things a first-time buyer should gather before applying.", stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:   # the last update repeats the full text; skip it
        seen += update.text
        print(redact(update.text), end="", flush=True)
final = await stream.get_final_response()
print("\n\n[usage] total tokens:", (final.usage_details or {}).get("total_token_count"))

A

 first

-time

 buyer

 should

 gather

... (28 outputs left out)

. Let me know if

 you want a full document checklist

.



[usage] total tokens: 264


### 2.4 Compliance check
The system prompt should hold even when the user pushes.

In [5]:
r = await agent.run("Promise me a 4.1% rate for 5 years, I want it in writing.")
print(redact(r.text))

I can provide an indicative rate quote for a 5-year mortgage, but I cannot promise or guarantee a specific rate, including 4.1%. Rates depend on your full application, credit review, and underwriting approval. Would you like me to check an indicative rate based on some basic information? 

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.


### 2.5 Session = conversation state
Each `run` is stateless unless you pass an `AgentSession`; the session carries history so follow-ups make sense.

In [6]:
session = agent.create_session()
r1 = await agent.run("I earn 120k and want to buy a 500k home with 100k down. Is that a reasonable pre-qual?", session=session)
print("Turn 1:", redact(r1.text)[:400], "\n")
r2 = await agent.run("What if I only had 50k down?", session=session)
print("Turn 2 (uses turn 1 context):", redact(r2.text)[:400])
print("\nsession id:", session.session_id)

Turn 1: With a $120k income and $100k down on a $500k home, you may be in a reasonable range for pre-qualification. Your down payment is 20%, which helps. The key factors next are your credit score, debts, and overall financial picture.

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval. 



Turn 2 (uses turn 1 context): With $50k down on a $500k home, your down payment would be 10%. This is often still eligible but may require mortgage insurance, which adds to cost. A lower down payment means higher monthly payments and stricter credit requirements.

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.

session id: 545c69cf-7aec-405d-8096-95561bbfa419


### 2.6 A tool
Any Python function becomes a tool with `tool(...)`: the model reads its name, type hints and docstring. Notebook 02 goes deep; here is the minimum.

In [7]:
from agent_framework import tool
from shared.fsi_common import get_rate_quote          # mock, fictional rate sheet

quote_tool = tool(get_rate_quote, approval_mode="never_require")
tool_agent = Agent(client, MAPLE_INSTRUCTIONS, name="maple-tools", tools=[quote_tool])
r = await tool_agent.run("Indicative 5-year fixed rate for a good-credit borrower at 75% LTV?")
for m in r.messages:
    for c in m.contents:
        if c.type == "function_call":   print("CALL  ", c.name, c.arguments)
        if c.type == "function_result": print("RESULT", str(c.result)[:120])
print(redact(r.text))

CALL   get_rate_quote {"credit_band":"good","ltv_pct":75,"term_yearys":5}
RESULT Error: Argument parsing failed.
CALL   get_rate_quote {"credit_band":"good","ltv_pct":75,"term_years":5}
RESULT {"rate_pct": 4.79, "apr_pct": 4.86, "term_years": 5, "as_of": "2026-11-01", "indicative": true, "disclosure": "Indicativ
It looks like there was a small error in processing the term years. Let me correct that and get the indicative 5-year fixed rate for a good-credit borrower at 75% LTV.The indicative 5-year fixed mortgage rate for a good-credit borrower at 75% LTV is approximately 4.79%. 

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.


### 2.7 A context provider
A `ContextProvider` runs `before_run` on every turn and can add instructions, messages or tools. Memory (nb04) and knowledge (nb03) are context providers. Here is a tiny one that injects today's rate-sheet date, so the model never invents it.

In [8]:
from agent_framework import ContextProvider
from shared.fsi_common import _AS_OF

class RateSheetDate(ContextProvider):
    def __init__(self):
        super().__init__("rate-sheet-date")
    async def before_run(self, *, agent, session, context, state):
        context.extend_instructions(self.source_id, f"The current rate sheet is dated {_AS_OF}. Quote this date when you mention rates.")

ctx_agent = Agent(client, MAPLE_INSTRUCTIONS, name="maple-ctx", tools=[quote_tool], context_providers=[RateSheetDate()])
r = await ctx_agent.run("Quote a 3-year fixed for excellent credit at 70% LTV and say how fresh the rate sheet is.")
print(redact(r.text))

For an excellent credit borrower with a 70% loan-to-value ratio, the indicative 3-year fixed rate is 4.79% with an APR of 4.86%. This rate is based on the rate sheet dated 2026-11-01, so it is very current. 

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.


## 3. Part B - Register the agent with Foundry (`AIProjectClient`)
So far the agent lived in our Python process. `AIProjectClient` is the **control plane** of your Foundry project: it stores, versions and lists server-side assets.

**What can be registered in a Foundry project?** (and where it is covered)

| Asset | SDK | Where |
|---|---|---|
| Prompt agent + versions (model, instructions, tools) | `pc.agents.create_version` | **this notebook**, nb02 |
| Hosted agent versions (your container) | `pc.agents.create_version(HostedAgentDefinition)` | nb07 |
| Tools: function schema, MCP, toolbox | `FunctionTool`, `MCPTool`, `pc.toolboxes` | nb02 |
| Connections (Search, App Insights, MCP auth) | `pc.connections` (read) / ARM (create) | nb02, nb03 |
| Knowledge bases / sources (Foundry IQ) | Azure AI Search + connection | nb03 |
| Memory stores | `pc.beta.memory_stores` | nb04 |
| Skills | `pc.beta.skills` | nb05 |
| Evaluators, eval runs, datasets | `pc.beta.evaluators`, `client.evals` | nb06 |

**Not registerable (stay in your code):** AF `Agent` objects, sessions (client-side), context providers, `@tool` function *bodies*, local MCP servers and local skill folders. Foundry only stores a *description* (e.g. a function schema); your code supplies the behaviour.

In [9]:
from azure.ai.projects import AIProjectClient

pc = AIProjectClient(endpoint=settings.project_endpoint, credential=credential, allow_preview=True)
print("project client ready; agents already in project:", sorted(a.name for a in pc.agents.list()))

project client ready; agents already in project: ['maple-hosted', 'maple-learn-mcp', 'maple-policy-iq', 'maple-prompt']


### 3.1 Create a prompt agent
A **prompt agent** = name + model + instructions (+ tools), stored server-side. Every `create_version` call adds a new **immutable** version, so you can roll back.

In [10]:
from azure.ai.projects.models import PromptAgentDefinition

AGENT = "nb01-maple-prompt"
v1 = pc.agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(model=settings.model, instructions=MAPLE_INSTRUCTIONS),
    description="Maple Mortgage Assistant (prompt agent, Day 2 nb01)",
    metadata={"day": "2", "notebook": "01"},
)
print("registered:", v1.name, "version", v1.version)

registered: nb01-maple-prompt version 1


### 3.2 List and get it back
`agents.list()` shows agents, `list_versions` the version history, `get_version` the stored definition. In the portal: **Foundry project -> Build -> Agents -> `nb01-maple-prompt`**.

In [11]:
# a second version with one extra rule -> immutable version history
v2 = pc.agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(model=settings.model, instructions=MAPLE_INSTRUCTIONS + "\n- Answer in at most 3 sentences."),
    description="v2: shorter answers",
)
print("agents:", sorted(a.name for a in pc.agents.list()))
for v in pc.agents.list_versions(agent_name=AGENT):
    print(f"  version {v['version']}  {v['description']}")
got = pc.agents.get_version(agent_name=AGENT, agent_version=v1.version)
print("\nget_version ->", got["definition"]["kind"], got["definition"]["model"], "| instructions start:", got["definition"]["instructions"][:60])

agents: ['maple-hosted', 'maple-learn-mcp', 'maple-policy-iq', 'maple-prompt', 'nb01-maple-prompt']
  version 2  v2: shorter answers
  version 1  Maple Mortgage Assistant (prompt agent, Day 2 nb01)



get_version -> prompt gpt-4.1-mini | instructions start: You are the Maple Mortgage Assistant for Northwind Home Lend


### 3.3 Call the registered agent
`FoundryAgent` is the AF wrapper for an agent that lives in Foundry: same `run`/stream API as in Part A, but the model call and instructions come from the server.

In [12]:
from agent_framework.foundry import FoundryAgent

remote = FoundryAgent(project_endpoint=settings.project_endpoint, agent_name=AGENT, credential=credential)
stream = remote.run("Pre-qual in two sentences, and can you guarantee 4.1%?", stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:
        seen += update.text
        print(redact(update.text), end="", flush=True)
await stream.get_final_response()
print()

I

 can

 help

 pre

-

qual

... (26 outputs left out)

 to lend. Rates and

 eligibility are subject to full

 application, credit adjudication

 and underwriter approval.

## 4. Hosted agents: what and when
A **hosted agent** is *your own agent code*, packaged as a container, deployed to and run by Foundry (managed scaling, identity, session state, endpoint), speaking the **Responses protocol**. Notebook 07 deploys one.

| | Code agent (Part A) | Prompt agent (Part B) | Hosted agent |
|---|---|---|---|
| Where code runs | your machine / app | Foundry (no code) | Foundry container (your code) |
| Defined by | Python | name + model + instructions + tools | container image |
| Registered in Foundry | no | yes (`PromptAgentDefinition`) | yes (`HostedAgentDefinition`, nb07) |
| Custom logic / libraries | full | none (built-in/MCP tools only) | full |
| Pick it when | prototyping, embedding in an app | single-model assistant with built-in/MCP tools | multi-step logic, custom tools, controlled code path |

**Rule of thumb:** start with a prompt agent; move to hosted when you need code the portal can't express.

### Tiny local run of the hosted-agent shape
The hosted entrypoint is just an AF `Agent` handed to `ResponsesHostServer`, which serves the OpenAI Responses protocol on port 8088. We write that file here, run it locally and call it (nb07 containerizes and registers it).

In [13]:
import subprocess, tempfile, textwrap, time, json, urllib.request

server_code = textwrap.dedent(f"""
    import os, sys
    sys.path.insert(0, r"{pathlib.Path.cwd().parent}")
    from azure.identity import DefaultAzureCredential
    from agent_framework import Agent
    from agent_framework.foundry import FoundryChatClient
    from agent_framework_foundry_hosting import ResponsesHostServer
    from shared.fsi_common import MAPLE_INSTRUCTIONS

    client = FoundryChatClient(project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
                               model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"], credential=DefaultAzureCredential())
    ResponsesHostServer(Agent(client, MAPLE_INSTRUCTIONS, name="maple-hosted")).run()
""")
script = pathlib.Path(tempfile.mkdtemp()) / "hosted_demo.py"
script.write_text(server_code)
proc = subprocess.Popen([sys.executable, str(script)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    body = json.dumps({"input": "In one sentence: what does mortgage pre-qualification mean?"}).encode()
    for _ in range(40):
        time.sleep(1)
        try:
            req = urllib.request.Request("http://127.0.0.1:8088/responses", body, {"Content-Type": "application/json"})
            out = json.load(urllib.request.urlopen(req, timeout=60))
            break
        except OSError:
            continue
    text = "".join(c.get("text", "") for o in out.get("output", []) for c in o.get("content", []))
    print("HTTP /responses ->", redact(text))
finally:
    proc.terminate()

HTTP /responses -> Mortgage pre-qualification is a preliminary assessment by a lender of a borrower's financial situation to estimate how much they might be eligible to borrow.


## 5. Clean up what this notebook registered
`agents.delete` removes the agent and all its versions (deleting the last version also removes the agent). Verify with `list`.

In [14]:
pc.agents.delete(agent_name=AGENT)
print("agents now:", sorted(a.name for a in pc.agents.list()))
assert AGENT not in [a.name for a in pc.agents.list()]

agents now: ['maple-hosted', 'maple-learn-mcp', 'maple-policy-iq', 'maple-prompt']


## Key takeaways
- AF builds the agent (client + `Agent` + tools + context providers + session) in *your* process; nothing is stored in Foundry.
- `AIProjectClient.agents.create_version` registers a **prompt agent**: versioned, immutable, visible in the portal, callable via `FoundryAgent`.
- Tool bodies, sessions, context providers and local skills/MCP servers are code, not Foundry assets.
- Hosted agents = your code, run by Foundry; choose them when prompt agents are not expressive enough.

**Next up:** [02-tools-and-mcp.ipynb](02-tools-and-mcp.ipynb) - give the agent tools and MCP servers.